In [1]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

In [3]:
books = pd.read_csv(r"C:\Users\inter\Desktop\Shrutika\ML Project\Book Recomendation(Collabroative Clustering)\Books.csv", low_memory=False)
print(books.head())
users = pd.read_csv(r"C:\Users\inter\Desktop\Shrutika\ML Project\Book Recomendation(Collabroative Clustering)\Users.csv", low_memory=False)
print(users.head())
ratings = pd.read_csv(r"C:\Users\inter\Desktop\Shrutika\ML Project\Book Recomendation(Collabroative Clustering)\Ratings.csv", low_memory=False)
print(ratings.head())

         ISBN                                         Book-Title  \
0  0195153448                                Classical Mythology   
1  0002005018                                       Clara Callan   
2  0060973129                               Decision in Normandy   
3  0374157065  Flu: The Story of the Great Influenza Pandemic...   
4  0393045218                             The Mummies of Urumchi   

            Book-Author Year-Of-Publication                   Publisher  \
0    Mark P. O. Morford                2002     Oxford University Press   
1  Richard Bruce Wright                2001       HarperFlamingo Canada   
2          Carlo D'Este                1991             HarperPerennial   
3      Gina Bari Kolata                1999        Farrar Straus Giroux   
4       E. J. W. Barber                1999  W. W. Norton &amp; Company   

                                         Image-URL-S  \
0  http://images.amazon.com/images/P/0195153448.0...   
1  http://images.amazon.com/

In [4]:
x = ratings['User-ID'].value_counts() > 200
active_users = x[x].index

filtered_rating = ratings[
    ratings['User-ID'].isin(active_users)
]

In [5]:
y = filtered_rating['ISBN'].value_counts() >= 50
famous_books = y[y].index

final_ratings = filtered_rating[
    filtered_rating['ISBN'].isin(famous_books)
]

In [6]:
final_df = final_ratings.merge(
    books,
    on='ISBN'
)

In [7]:
pivot_table = final_df.pivot_table(
    index='Book-Title',
    columns='User-ID',
    values='Book-Rating'
)

In [8]:
pivot_table.fillna(0, inplace=True)

User-ID,254,2276,2766,2977,3363,4017,4385,6242,6251,6323,...,274004,274061,274301,274308,274808,275970,277427,277478,277639,278418
Book-Title,,,,,,,,,,,,,,,,,,,,,
1984,9.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1st to Die: A Novel,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2nd Chance,0.0,10.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4 Blondes,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
A Bend in the Road,0.0,0.0,7.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
Year of Wonders,0.0,0.0,0.0,7.0,0.0,0.0,0.0,7.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
You Belong To Me,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Zen and the Art of Motorcycle Maintenance: An Inquiry into Values,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [9]:
similarity_scores = cosine_similarity(
    pivot_table
)

In [10]:
def recommend_book(book_name, top_n=5):

    if book_name not in pivot_table.index:
        return ["Book not found"]

    index = np.where(
        pivot_table.index == book_name
    )[0][0]

    similar_books = sorted(
        list(enumerate(similarity_scores[index])),
        key=lambda x: x[1],
        reverse=True
    )[1:top_n+1]

    recommendations = []

    for i in similar_books:

        temp_df = books[
            books['Book-Title'] ==
            pivot_table.index[i[0]]
        ]

        book_details = temp_df.drop_duplicates(
            'Book-Title'
        )[[
            'Book-Title',
            'Book-Author',
            'Publisher',
            'Year-Of-Publication'
        ]]

        recommendations.append(
            book_details.values.tolist()[0]
        )

    return recommendations

In [11]:
book_name = "1984"

In [12]:
result = recommend_book(book_name)

In [13]:
for idx, rec in enumerate(result, start=1):
    print(f"\nRecommendation {idx}")
    print("Title     :", rec[0])
    print("Author    :", rec[1])
    print("Publisher :", rec[2])
    print("Year      :", rec[3])


Recommendation 1
Title     : The Handmaid's Tale
Author    : Margaret Atwood
Publisher : Fawcett Books
Year      : 1989

Recommendation 2
Title     : The Catcher in the Rye
Author    : J.D. Salinger
Publisher : Little, Brown
Year      : 1991

Recommendation 3
Title     : Slaughterhouse Five or the Children's Crusade: A Duty Dance With Death
Author    : Kurt Vonnegut
Publisher : Laurel
Year      : 1991

Recommendation 4
Title     : The Vampire Lestat (Vampire Chronicles, Book II)
Author    : ANNE RICE
Publisher : Ballantine Books
Year      : 1986

Recommendation 5
Title     : The Tale of the Body Thief (Vampire Chronicles (Paperback))
Author    : Anne Rice
Publisher : Ballantine Books
Year      : 1993


In [16]:
import pickle 

pickle.dump(
    books,
    open('books.pkl', 'wb')
)
pickle.dump(
    pivot_table,
    open('pivot_table.pkl', 'wb')
)
pickle.dump(
    similarity_scores,
    open('similarity_scores.pkl', 'wb')
)

In [17]:
import pickle

books = pickle.load(
    open(r'C:\Users\inter\Desktop\Shrutika\ML Project\Book Recomendation(Collabroative Clustering)\books.pkl', 'rb')
)

pivot_table = pickle.load(
    open(r'C:\Users\inter\Desktop\Shrutika\ML Project\Book Recomendation(Collabroative Clustering)\pivot_table.pkl', 'rb')
)

similarity_scores = pickle.load(
    open(r'C:\Users\inter\Desktop\Shrutika\ML Project\Book Recomendation(Collabroative Clustering)\similarity_scores.pkl', 'rb')
)

In [18]:
import numpy as np

def recommend(book_name):

    if book_name not in pivot_table.index:
        return ["Book not found"]

    index = np.where(
        pivot_table.index == book_name
    )[0][0]

    similar_items = sorted(
        list(enumerate(similarity_scores[index])),
        key=lambda x: x[1],
        reverse=True
    )[1:6]

    data = []

    for i in similar_items:

        temp_df = books[
            books['Book-Title'] ==
            pivot_table.index[i[0]]
        ]

        item = temp_df.drop_duplicates(
            'Book-Title'
        )[[
            'Book-Title',
            'Book-Author',
            'Publisher',
            'Year-Of-Publication'
        ]]

        data.append(
            item.values.tolist()[0]
        )

    return data

In [20]:
result = recommend("1984")

for idx, rec in enumerate(result, start=1):
    print(f"\nRecommendation {idx}")
    print("Title     :", rec[0])
    print("Author    :", rec[1])
    print("Publisher :", rec[2])
    print("Year      :", rec[3])


Recommendation 1
Title     : The Handmaid's Tale
Author    : Margaret Atwood
Publisher : Fawcett Books
Year      : 1989

Recommendation 2
Title     : The Catcher in the Rye
Author    : J.D. Salinger
Publisher : Little, Brown
Year      : 1991

Recommendation 3
Title     : Slaughterhouse Five or the Children's Crusade: A Duty Dance With Death
Author    : Kurt Vonnegut
Publisher : Laurel
Year      : 1991

Recommendation 4
Title     : The Vampire Lestat (Vampire Chronicles, Book II)
Author    : ANNE RICE
Publisher : Ballantine Books
Year      : 1986

Recommendation 5
Title     : The Tale of the Body Thief (Vampire Chronicles (Paperback))
Author    : Anne Rice
Publisher : Ballantine Books
Year      : 1993
